In [1]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd

In [2]:
sys.path.append('/Users/stevie/repos/lingo_kit_data/vocabulary')
from stanza_tokenizer import StanzaTokenizer

/Users/stevie/repos/lingo_kit_data/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-11-15 23:02:48 INFO: Downloaded file to /Users/stevie/stanza_resources/resources.json
2025-11-15 23:02:48 INFO: Downloading default packages for language: it (Italian) ...
2025-11-15 23:02:49 INFO: File exists: /Users/stevie/stanza_resources/it/default.zip
2025-11-15 23:02:50 INFO: Finished downloading models and saved to /Users/stevie/stanza_resources


In [3]:
bad_pos = ['X', 'PUNCT', 'PROPN', 'NUM']

In [4]:
df = pd.read_csv('../sentence_dataframe.tsv', sep='\t')
len(df), df.columns

(624335, Index(['text_it', 'text_en', 'hash'], dtype='object'))

In [5]:
df['term_length_it'] = df['text_it'].str.split().apply(len)
df['term_length_en'] = df['text_en'].str.split().apply(len)
df['char_length_it'] = df['text_it'].str.len()
df['char_length_en'] = df['text_en'].str.len()

In [6]:
print(len(df))
df = df[df['char_length_it'] < 50]
print(len(df))

624335
577993


In [7]:
tokenizer = StanzaTokenizer()

2025-11-15 23:02:53 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2025-11-15 23:02:53 INFO: Downloaded file to /Users/stevie/stanza_resources/resources.json
2025-11-15 23:02:53 WARNING: Language it package default expects mwt, which has been added
2025-11-15 23:02:54 INFO: Loading these models for language: it (Italian):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |
| depparse  | combined_charlm   |

2025-11-15 23:02:54 INFO: Using device: cpu
2025-11-15 23:02:54 INFO: Loading: tokenize
2025-11-15 23:02:55 INFO: Loading: mwt
2025-11-15 23:02:55 INFO: Loading: pos
2025-11-15 23:02:56 INFO: Loading: lemma
2025-11-15 23:02:56 INFO: Loading: depparse
2025-11-15 23:02:56 INFO: Done loading 

In [8]:
def get_token_key(token_dict):
    return f"{token_dict['text']}_{token_dict['lemma']}_{token_dict['pos']}"

In [ ]:
gram2_dict = {}
iter_df = df.sample(frac=1, random_state=42).reset_index(drop=True)
# iter_df = df.sample(n=5000, random_state=42).reset_index(drop=True)
for sentence in tqdm(iter_df['text_it'], total=len(iter_df)):
    start_word_count = {}
    end_word_count = {}
    tokens = tokenizer.tokenize(sentence)
    for i in range(len(tokens)-2):
        t1 = tokens[i]
        t2 = tokens[i+1]

        # keep track of the index of 'find' words
        if t1['text'].lower() not in start_word_count:
            start_word_count[t1['text'].lower()] = -1
        start_word_count[t1['text'].lower()] += 1
        if t2['text'].lower() not in end_word_count:
            end_word_count[t2['text'].lower()] = -1
        end_word_count[t2['text'].lower()] += 1

        if t1['pos'] in bad_pos or t2['pos'] in bad_pos:
            continue

        gram2 = (get_token_key(t1), get_token_key(t2))
        
        start_index = sentence.lower().find(t1['text'].lower(), start_word_count[t1['text'].lower()])
        end_index = sentence.lower().find(t2['text'].lower(), end_word_count[t2['text'].lower()]) + len(t2['text'])
        substr = sentence[start_index:end_index]

        if gram2 not in gram2_dict:
            gram2_dict[gram2] = []

        gram2_dict[gram2].append({
            'sentence': sentence,
            'substr': substr,
            'start_index': start_index,
            'end_index': end_index,
            'token_1': t1,
            'token_2': t2,
        })

100%|██████████| 5000/5000 [06:26<00:00, 12.93it/s]


In [11]:
gram2_items = sorted(gram2_dict.items(), key=lambda x: len(x[1]), reverse=True)
for i in range(20):
    print(gram2_items[i][1][0]['token_1']['text'])
    print(gram2_items[i][1][0]['token_2']['text'])
    for j, t in enumerate(gram2_items[i][1]):
        print(t['substr'])
        if j > 2:
            break
        # break

il
suo
il suo
il suo
il suo
Il suo
ha
detto
ha detto
ha detto
Ha detto
ha detto
non
è
non è
non è
Non è
Non è
detto
che
detto che
detto che
detto che
detto che
io
non
Io non
Io non
Io non
Io non
la
sua
La sua
la sua
la sua
la sua
io
sono
Io sono
Io sono
Io sono
Io sono
a
casa
a casa
Adesso vada a casa
andata a casa
asciarono i loro passaporti a casa
è
un
È un
è un
è un
è un
è
il
è il
è il
è il
è il
che
non
che non
che non
che non
che non
il
mio
il mio
il mio
il mio
il mio
la
mia
la mia
la mia
lare con la mia
La mia
non
mi
non mi
Non mi
Non mi
non mi
si
è
si è
si è
si è
si è
mi
ha
mi ha
mi ha
Mi ha
Mi ha
non
sono
Non sono
non sono
Non sono
Non sono
disse
che
disse che
Disse che
disse che
disse che
è
una
È una
è una
è una
È una
c'
è
C'è
c'è
C'è
C'è


In [12]:
len(gram2_dict)

12004

In [13]:
gram2_dict[list(gram2_dict.keys())[0]][0].keys()

dict_keys(['sentence', 'substr', 'start_index', 'end_index', 'token_1', 'token_2'])

In [14]:
data = {
    'gram2_key': [], 
    'count': [],
    'substr': [],
    'term_1': [],
    'term_2': [],
    'lemma_1': [],
    'lemma_2': [],
    'pos_1': [],
    'pos_2': [],
}
new_gram2_dict = {}
len(gram2_dict)
for gram_key, gram_list in gram2_dict.items():
    substr_votes = {}
    if len(gram_list) < 5:
        continue
    for entry in gram_list:
        substr = entry['substr']
        if substr not in substr_votes:
            substr_votes[substr] = 0
        substr_votes[substr] += 1
    sorted_substrs = sorted(substr_votes.items(), key=lambda x: x[1], reverse=True)
    best_substr, best_votes = sorted_substrs[0]
    data['gram2_key'].append(gram_key)
    data['substr'].append(best_substr.lower())
    data['term_1'].append(gram_key[0].split('_')[0])
    data['term_2'].append(gram_key[1].split('_')[0])
    data['lemma_1'].append(gram_key[0].split('_')[1])
    data['lemma_2'].append(gram_key[1].split('_')[1])
    data['pos_1'].append(gram_key[0].split('_')[2])
    data['pos_2'].append(gram_key[1].split('_')[2])
    data['count'].append(len(gram_list))
gram2_df = pd.DataFrame(data)

In [15]:
len(gram2_df)

438

In [16]:
gram2_df.sort_values(by='count', ascending=False, inplace=True)

In [17]:
gram2_df.to_csv('gram2.csv', sep='\t', index=False)